## Assignment: Diophantine Equations

> This assignment is part of [AI for Beginners Curriculum](http://github.com/microsoft/ai-for-beginners) and is inspired by [this post](https://habr.com/post/128704/).

Your goal is to solve so-called **Diophantine equation** - an equation with integer roots and integer coefficients. For example, consider the following equation:

$$a+2b+3c+4d=30$$

You need to find integer roots $a$,$b$,$c$,$d\in\mathbb{N}$ that satisfy this equation.

Hints:
1. You can consider roots to be in the interval [0;30]
1. As a gene, consider using the list of root values

## Implementation

We will use genetic algorithm to find integer roots of the equation. Let's define the problem and implement the solution.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

### Problem Definition

Given the equation: $a + 2b + 3c + 4d = 30$

We need to find integer values for $a, b, c, d \in [0, 30]$ that satisfy this equation.

**Gene representation**: Each gene is a list of 4 integers `[a, b, c, d]` representing the four roots.

In [ ]:
# Coefficients and target value
coeffs = [1, 2, 3, 4]
target = 30
max_val = 30  # Maximum value for each root

def generate_gene():
    """Generate a random gene (solution candidate)"""
    return [random.randint(0, max_val) for _ in range(len(coeffs))]

def fit(gene):
    """Fitness function: absolute difference from target value"""
    result = sum(c * g for c, g in zip(coeffs, gene))
    return abs(result - target)

# Test
test_gene = generate_gene()
print(f"Random gene: {test_gene}, result: {sum(c*g for c,g in zip(coeffs, test_gene))}, fitness: {fit(test_gene)}")

### Genetic Operators

In [ ]:
def mutate(gene):
    """Mutation: change one random position to a new random value"""
    new_gene = gene.copy()
    idx = random.randint(0, len(new_gene) - 1)
    new_gene[idx] = random.randint(0, max_val)
    return new_gene

def crossover(gene1, gene2):
    """Crossover: combine two genes at a random point"""
    point = random.randint(1, len(gene1) - 1)
    return gene1[:point] + gene2[point:]

# Test
g1 = generate_gene()
g2 = generate_gene()
print(f"Gene 1: {g1}")
print(f"Gene 2: {g2}")
print(f"Mutation of g1: {mutate(g1)}")
print(f"Crossover: {crossover(g1, g2)}")

### Evolution Algorithm

In [ ]:
def solve_diophantine(pop_size=50, max_generations=5000, mutation_prob=0.3):
    """
    Solve Diophantine equation using genetic algorithm.
    Returns the solution gene and fitness history.
    """
    # Initialize population
    population = [generate_gene() for _ in range(pop_size)]
    history = []
    
    for gen in range(max_generations):
        # Calculate fitness for all genes
        fitnesses = [(gene, fit(gene)) for gene in population]
        fitnesses.sort(key=lambda x: x[1])
        
        best_gene, best_fit = fitnesses[0]
        history.append(best_fit)
        
        # Check if solution found
        if best_fit == 0:
            return best_gene, history, gen + 1
        
        # Selection: keep top 50%
        population = [g for g, f in fitnesses[:pop_size // 2]]
        
        # Create new generation
        while len(population) < pop_size:
            # Select two parents (with bias toward better fitness)
            idx1 = random.randint(0, len(population) // 2 - 1)
            idx2 = random.randint(0, len(population) // 2 - 1)
            
            # Crossover
            child = crossover(population[idx1], population[idx2])
            
            # Mutation
            if random.random() < mutation_prob:
                child = mutate(child)
            
            population.append(child)
    
    # Return best found (not exact solution)
    return best_gene, history, max_generations

# Solve
solution, history, generations = solve_diophantine()
print(f"Solution found in {generations} generations: {solution}")
print(f"Verification: {coeffs[0]}*{solution[0]} + {coeffs[1]}*{solution[1]} + {coeffs[2]}*{solution[2]} + {coeffs[3]}*{solution[3]} = {sum(c*g for c,g in zip(coeffs, solution))}")

### Convergence Visualization

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history)
plt.xlabel('Generation')
plt.ylabel('Best Fitness')
plt.title('Genetic Algorithm Convergence')
plt.grid(True)
plt.show()

### Finding Multiple Solutions

The Diophantine equation may have multiple solutions. Let's find several of them.

In [ ]:
def find_multiple_solutions(n_solutions=10, max_attempts=100):
    """Find multiple distinct solutions"""
    solutions = set()
    attempts = 0
    
    while len(solutions) < n_solutions and attempts < max_attempts:
        sol, _, _ = solve_diophantine()
        solutions.add(tuple(sol))
        attempts += 1
    
    return [list(s) for s in solutions]

print("Multiple solutions found:")
solutions = find_multiple_solutions(10)
for i, sol in enumerate(solutions, 1):
    result = sum(c * g for c, g in zip(coeffs, sol))
    print(f"{i}. a={sol[0]}, b={sol[1]}, c={sol[2]}, d={sol[3]} → {coeffs[0]}*{sol[0]} + {coeffs[1]}*{sol[1]} + {coeffs[2]}*{sol[2]} + {coeffs[3]}*{sol[3]} = {result}")

### Extending to Other Equations

The genetic algorithm approach can be easily extended to solve other Diophantine equations by modifying the coefficients and target value.

In [ ]:
def solve_general_diophantine(coefficients, target_val, pop_size=100, max_generations=10000):
    """
    Solve any Diophantine equation using genetic algorithm.
    Example: solve_general_diophantine([2, 3, 5], 50) solves 2a + 3b + 5c = 50
    """
    global coeffs, target, max_val
    coeffs = coefficients
    target = target_val
    max_val = target_val  # Upper bound for roots
    
    return solve_diophantine(pop_size, max_generations)

# Example: 2a + 3b + 5c = 50
sol, hist, gen = solve_general_diophantine([2, 3, 5], 50)
print(f"Solution for 2a + 3b + 5c = 50: a={sol[0]}, b={sol[1]}, c={sol[2]}")
print(f"Verification: 2*{sol[0]} + 3*{sol[1]} + 5*{sol[2]} = {2*sol[0] + 3*sol[1] + 5*sol[2]}")